# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoamenAbouhaty/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [31]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connection ready.")

DuckDB connection ready.


In [32]:
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

con.sql(f"""
DESCRIBE SELECT *
FROM {QUERY_90D}
""").show()

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_hash_id                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_char_count              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ query_token_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ window_start                  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ window_end                    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ impressions_90d               

In [33]:
con.sql(f"""
SELECT
    MIN(window_start) AS min_window_start,
    MAX(window_end) AS max_window_end,
    COUNT(*) AS row_count
FROM {QUERY_90D}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬────────────────┬───────────┐
│ min_window_start │ max_window_end │ row_count │
│       date       │      date      │   int64   │
├──────────────────┼────────────────┼───────────┤
│ 2026-04-02       │ 2026-06-30     │   2414248 │
└──────────────────┴────────────────┴───────────┘



## 1. Unit of analysis + time window

### Unit of analysis

One row represents a single search query for a single content item for a single client:

`client_hash_id × content_hash_id × query_hash_id`

The source table is `fact_content_query_90d`.

The table contains aggregated search-performance metrics over a fixed 90-day window, including impressions, clicks, average position, and query visibility signals.

### Time window

The analysis uses the fixed 90-day window provided by the dataset, defined by `window_start` and `window_end`.

The window is treated as historical search-performance context available at the decision point.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields

**Features:**  
- `query_char_count` — query length in characters.
- `query_token_count` — query length in tokens.
- `impressions_90d` — historical query impressions.
- `clicks_90d` — historical query clicks.
- `avg_position_90d` — historical average search position.

**Label:**

- A future search performance outcome is not available directly in this fixed 90-day query snapshot. A supervised label would need to be defined from a separate, later observation window.

**Context:**  
- `client_hash_id`
- `content_hash_id`
- `query_hash_id`
- `window_start`
- `window_end`
- `content_visible_query_count`
- `rare_query_count`
- `rare_impressions_share`
- `anonymized_impressions_share`

**Excluded:**  
- Any information that is only known after the decision point or is derived from the future outcome, because it would introduce target leakage.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


In [36]:
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

con.sql(f"""
DESCRIBE SELECT *
FROM {DAILY}
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [37]:
con.sql(f"""
SELECT column_name, column_type
FROM (
    DESCRIBE SELECT *
    FROM {DAILY}
)
ORDER BY column_name
""").show(max_rows=100)

┌──────────────────────────┬─────────────┐
│       column_name        │ column_type │
│         varchar          │   varchar   │
├──────────────────────────┼─────────────┤
│ ai_chatgpt               │ BIGINT      │
│ ai_claude                │ BIGINT      │
│ ai_copilot               │ BIGINT      │
│ ai_gemini                │ BIGINT      │
│ ai_meta                  │ BIGINT      │
│ ai_other                 │ BIGINT      │
│ ai_perplexity            │ BIGINT      │
│ client_has_ga4           │ BOOLEAN     │
│ client_has_gsc           │ BOOLEAN     │
│ client_hash_id           │ VARCHAR     │
│ content_hash_id          │ VARCHAR     │
│ ga4_data_available       │ BOOLEAN     │
│ ga4_engaged_sessions     │ BIGINT      │
│ ga4_pageviews            │ BIGINT      │
│ ga4_sessions             │ BIGINT      │
│ ga4_total_engagement_sec │ BIGINT      │
│ ga4_users                │ BIGINT      │
│ gsc_avg_position         │ DOUBLE      │
│ gsc_clicks               │ BIGINT      │
│ gsc_data_

In [38]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
        AS unique_client_content_days
FROM {DAILY}
WHERE month = '2026-03'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────────────┐
│ total_rows │ unique_client_content_days │
│   int64    │           int64            │
├────────────┼────────────────────────────┤
│    9841378 │                    9841378 │
└────────────┴────────────────────────────┘



In [39]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {DAILY}
WHERE month = '2026-03'
""").show()

┌───────────┬─────────────────┬─────────────────┐
│ row_count │ min_report_date │ max_report_date │
│   int64   │      date       │      date       │
├───────────┼─────────────────┼─────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │
└───────────┴─────────────────┴─────────────────┘



In [40]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM {DAILY}
WHERE month = '2026-03'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Five candidate features

1. `query_char_count` — knowable at the decision moment because it is part of the historical query snapshot.

2. `query_token_count` — knowable at the decision moment because it is part of the historical query snapshot.

3. `impressions_90d` — knowable at the decision moment because it summarizes historical impressions within the available 90-day window.

4. `clicks_90d` — knowable at the decision moment because it summarizes historical clicks within the available 90-day window.

5. `avg_position_90d` — knowable at the decision moment because it summarizes historical search position within the available 90-day window.

### Data limits

- The dataset contains pseudonymized client, content, and query identifiers, so it cannot provide real client names, domains, URLs, titles, or raw query text.

- GSC and GA4 availability is not uniform across all client-content-day rows. Therefore, missing data can represent unavailable measurement rather than zero activity.

- The `fact_content_query_90d` table represents a fixed historical 90-day window, so it does not by itself provide a future outcome for supervised learning.

- Time windows must be kept separate. Features must be calculated only from information available before the decision cutoff, while labels must come from a subsequent period.

- The dataset does not explain the causal reason for a change in search performance; it can show observed patterns but cannot establish causality.

### Leakage trap

To demonstrate target leakage, we intentionally add a feature derived directly from the label. A model using this leaked feature should achieve an unrealistically strong score.

The leaked feature is removed afterward. The final feature set contains only information available at the decision moment.

---

**Leakage result:** The intentionally leaked feature directly encoded the label, producing a perfect accuracy of 1.0000. This demonstrates how target leakage can create an unrealistically strong result. The leaked feature was removed and is not part of the final feature set.

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [43]:
# Intentional leakage demonstration
# The leaked feature is derived directly from the label.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df_leak = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    CASE
        WHEN gsc_clicks > 0 THEN 1
        ELSE 0
    END AS label
FROM {DAILY}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_impressions > 0
LIMIT 100000
""").df()

# Deliberate leak: feature is exactly the label.
df_leak["leaked_feature"] = df_leak["label"]

X = df_leak[["leaked_feature"]]
y = df_leak["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)

leaked_accuracy = accuracy_score(
    y_test,
    model.predict(X_test)
)

print(f"Leaked accuracy: {leaked_accuracy:.4f}")
print("The leaked feature is removed and must not be used in the final feature set.")

Leaked accuracy: 1.0000
The leaked feature is removed and must not be used in the final feature set.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.